In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [12]:
df = pd.read_csv("jan to may police violation_anonymized791b166.csv")
df.head()

,id,latitude,longitude,location,vehicle_number,vehicle_type,description,violation_type,offence_code,created_datetime,...,center_code,police_station,data_sent_to_scita,junction_name,action_taken_timestamp,data_sent_to_scita_timestamp,updated_vehicle_number,updated_vehicle_type,validation_status,validation_timestamp
0,FKID000000,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,CAR,NaN,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0000,MAXI-CAB,approved,2023-11-30 03:08:24.818+00
1,FKID000001,12.905463,77.700778,"Sarjapura Main Road, The Grove, Janatha Colony...",FKN00GL0001,CAR,NaN,"[""NO PARKING""]",[113],2023-11-24 22:46:46+00,...,82.0,Bellandur,False,No Junction,NaN,NaN,NaN,NaN,NaN,NaN
2,FKID000002,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,CAR,NaN,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,...,9.0,Madiwala,True,No Junction,NaN,NaN,FKN00GL0002,MAXI-CAB,approved,2023-11-30 03:08:56.998+00
3,FKID000003,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,NaN,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00,...,26.0,Byatarayanapura,True,No Junction,NaN,NaN,FKN00GL0003,SCOOTER,approved,2023-11-18 23:35:12.428+00
4,FKID000004,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,NaN,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00,...,3.0,Upparpet,True,BTP044 - Sagar Theatre Junction,NaN,NaN,FKN00GL0004,TANKER,approved,2023-11-30 03:11:32.796+00


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 298450 entries, 0 to 298449
Data columns (total 24 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            298450 non-null  str    
 1   latitude                      298450 non-null  float64
 2   longitude                     298450 non-null  float64
 3   location                      295409 non-null  str    
 4   vehicle_number                298450 non-null  str    
 5   vehicle_type                  298450 non-null  str    
 6   description                   0 non-null       float64
 7   violation_type                298450 non-null  str    
 8   offence_code                  298450 non-null  str    
 9   created_datetime              298450 non-null  str    
 10  closed_datetime               0 non-null       float64
 11  modified_datetime             298450 non-null  str    
 12  device_id                     298450 non-null  str    


In [14]:
df.shape

(298450, 24)

In [15]:
df.isnull().sum()

id                                   0
latitude                             0
longitude                            0
location                          3041
vehicle_number                       0
vehicle_type                         0
description                     298450
violation_type                       0
offence_code                         0
created_datetime                     0
closed_datetime                 298450
modified_datetime                    0
device_id                            0
created_by_id                        5
center_code                      11260
police_station                       5
data_sent_to_scita                   0
junction_name                        5
action_taken_timestamp          298450
data_sent_to_scita_timestamp    256289
updated_vehicle_number          125254
updated_vehicle_type            125254
validation_status               125254
validation_timestamp            125254
dtype: int64

In [16]:
## Remove all the columns which has almost all null values

null_columns = ["id", "description", "closed_datetime", "action_taken_timestamp", "data_sent_to_scita_timestamp"]
df = df.drop(columns= null_columns)

In [17]:
## To predict the traffic hostspot we do not need traffic camera information, so we remove them as well

unimp_cols = ["data_sent_to_scita", "device_id", "created_by_id", "modified_datetime"]
df.drop(columns=unimp_cols, inplace=True)

In [18]:
## Camera sensors sometimes misreads the numbers. later the officials corrects it while patrolling. So we take all correct values only

df['vehicle_type'] = df['updated_vehicle_type'].fillna(df['vehicle_type'])
df['vehicle_number'] = df['updated_vehicle_number'].fillna(df['vehicle_number'])
df.drop(columns=['updated_vehicle_type', 'updated_vehicle_number'], inplace=True)

In [19]:
# Keep only valid, real traffic offenses
df = df[df['validation_status'] == 'approved']

In [20]:
df.shape

(115400, 13)

In [21]:
df.head()

,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,center_code,police_station,junction_name,validation_status,validation_timestamp
0,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,MAXI-CAB,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00,9.0,Madiwala,No Junction,approved,2023-11-30 03:08:24.818+00
2,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,MAXI-CAB,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00,9.0,Madiwala,No Junction,approved,2023-11-30 03:08:56.998+00
3,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00,26.0,Byatarayanapura,No Junction,approved,2023-11-18 23:35:12.428+00
4,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00,3.0,Upparpet,BTP044 - Sagar Theatre Junction,approved,2023-11-30 03:11:32.796+00
5,12.981677,77.608125,"Dispensary Road, Tasker Town, Shivaji Nagar, B...",FKN00GL0005,SCOOTER,"[""NO PARKING""]",[113],2023-11-28 07:04:46+00,16.0,Shivajinagar,BTP051 - Safina Plaza Junction,approved,2023-11-30 04:32:19.24+00


In [22]:
df.info()

<class 'pandas.DataFrame'>
Index: 115400 entries, 0 to 298449
Data columns (total 13 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   latitude              115400 non-null  float64
 1   longitude             115400 non-null  float64
 2   location              114852 non-null  str    
 3   vehicle_number        115400 non-null  str    
 4   vehicle_type          115400 non-null  str    
 5   violation_type        115400 non-null  str    
 6   offence_code          115400 non-null  str    
 7   created_datetime      115400 non-null  str    
 8   center_code           112956 non-null  float64
 9   police_station        115400 non-null  str    
 10  junction_name         115400 non-null  str    
 11  validation_status     115400 non-null  str    
 12  validation_timestamp  115400 non-null  str    
dtypes: float64(3), str(10)
memory usage: 37.5 MB


In [23]:
# Convert timed columns to pandas datetime objects
df['created_datetime'] = pd.to_datetime(df['created_datetime'], format='mixed')
df['validation_timestamp'] = pd.to_datetime(df['validation_timestamp'], format='mixed')

# Calculate how many minutes it took a human to approve the ticket
df['approval_delay_hours'] = (df['validation_timestamp'] - df['created_datetime']).dt.total_seconds() / 3600

In [25]:
from sklearn.cluster import DBSCAN
import numpy as np

# Select coordinates and convert to radians for precise geographical math
coords = df[['latitude', 'longitude']].dropna()
kms_per_radian = 6371.0088
epsilon_meters = 100  # Set the hotspot search radius (e.g., 100 meters)
epsilon_radians = (epsilon_meters / 1000) / kms_per_radian

# Run DBSCAN (Min 5 violations within 100 meters to form a hotspot)
db = DBSCAN(eps=epsilon_radians, min_samples=5, metric='haversine')
coords['hotspot_cluster'] = db.fit_predict(np.radians(coords))

#  Merge results back (Noise points will automatically be labeled as -1)
df = df.join(coords['hotspot_cluster'])

In [27]:
df.to_csv("hotspot_Data.csv")

In [29]:
df1 = pd.read_csv("hotspot_Data.csv")
df1.head()

,Unnamed: 0,latitude,longitude,location,vehicle_number,vehicle_type,violation_type,offence_code,created_datetime,center_code,police_station,junction_name,validation_status,validation_timestamp,approval_delay_hours,hotspot_cluster
0,0,12.925557,77.618665,"18th Main Road, Block 2, Koramangala, Bengalur...",FKN00GL0000,MAXI-CAB,"[""WRONG PARKING"",""PARKING NEAR ROAD CROSSING""]","[112,104]",2023-11-20 00:28:46+00:00,9.0,Madiwala,No Junction,approved,2023-11-30 03:08:24.818000+00:00,242.660783,0
1,2,12.925449,77.618504,"Koramangala 2nd Block, Kormangala West, Bengal...",FKN00GL0002,MAXI-CAB,"[""WRONG PARKING"",""PARKING IN A MAIN ROAD""]","[112,107]",2023-11-20 00:27:46+00:00,9.0,Madiwala,No Junction,approved,2023-11-30 03:08:56.998000+00:00,242.686388,0
2,3,12.956521,77.518618,"6th Cross Road, Manasa Layout, Nagarbhavi, Ben...",FKN00GL0003,SCOOTER,"[""NO PARKING""]",[113],2023-11-16 06:47:46+00:00,26.0,Byatarayanapura,No Junction,approved,2023-11-18 23:35:12.428000+00:00,64.790674,1
3,4,12.977767,77.580545,"Kalidasa Road, Gandhinagar, Nehru Nagar, Benga...",FKN00GL0004,TANKER,"[""NO PARKING""]",[113],2023-11-22 04:56:46+00:00,3.0,Upparpet,BTP044 - Sagar Theatre Junction,approved,2023-11-30 03:11:32.796000+00:00,190.246332,2
4,5,12.981677,77.608125,"Dispensary Road, Tasker Town, Shivaji Nagar, B...",FKN00GL0005,SCOOTER,"[""NO PARKING""]",[113],2023-11-28 07:04:46+00:00,16.0,Shivajinagar,BTP051 - Safina Plaza Junction,approved,2023-11-30 04:32:19.240000+00:00,45.459233,3


In [ ]:
import folium
from folium.plugins import MarkerCluster

# Filter out the "noise" (-1) so you only plot the true hotspots
hotspots = df[df['hotspot_cluster'] != -1]

# Initialize the map centered around Bangalore
map_bangalore = folium.Map(location=[12.9716, 77.5946], zoom_start=12)

# Create a fast marker cluster layer for the violations
marker_cluster = MarkerCluster().add_to(map_bangalore)

# Add each violation to the map with its junction name as a popup
for idx, row in hotspots.iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"Junction: {row['junction_name']}<br>Cluster: {row['hotspot_cluster']}",
        icon=folium.Icon(color='red', icon='info-sign')
    ).add_to(marker_cluster)

# Save and view the map
map_bangalore.save("hotspots.html")

In [ ]:
# Get the cluster IDs of the top 10 most crowded DBSCAN clusters
top_10_cluster_ids = df[df['hotspot_cluster'] != -1]['hotspot_cluster'].value_counts().head(10).index

# Filter your DataFrame to keep ONLY these top 10 high-priority zones
df_top_10 = df[df['hotspot_cluster'].isin(top_10_cluster_ids)]

In [ ]:
# Create a summary profile for your top 10 zones
top_10_profiles = df_top_10.groupby('hotspot_cluster').agg(
    total_violations=('hotspot_cluster', 'count'),
    primary_junction=('junction_name', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
    primary_police_station=('police_station', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
    dominant_violation=('violation_type', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
    vehicle_type=('vehicle_type', lambda x: x.mode()[0] if not x.mode().empty else "Unknown")
).sort_values(by='total_violations', ascending=False)

print(top_10_profiles)

                 total_violations                    primary_junction  \
hotspot_cluster                                                         
2                           26471             BTP040 - Elite Junction   
3                           10417      BTP051 - Safina Plaza Junction   
5                            7516                         No Junction   
29                           6318    BTP020 - Hosahalli Metro Station   
8                            6217                         No Junction   
31                           4020                         No Junction   
22                           2175  BTP016 - 5th Main Road, RPC Layout   
17                           1818                         No Junction   
15                           1612             BTP032 - Windsor Circle   
21                           1425                         No Junction   

                primary_police_station dominant_violation vehicle_type  
hotspot_cluster                                   

In [ ]:
df_top_10['hour'] = pd.to_datetime(df_top_10['validation_timestamp']).dt.hour
time_insights = df_top_10.groupby('hotspot_cluster')['hour'].agg(lambda x: x.mode()[0])
time_insights

hotspot_cluster
2      0
3     23
5      0
8      0
15     1
17    23
21    22
22     0
29    23
31    23
Name: hour, dtype: int32

In [ ]:
# 1. Filter data for standard daytime business hours (e.g., 8 AM to 8 PM)
df_daytime = df_top_10[df_top_10['hour'].between(8, 20)]

# 2. Get the top daytime clusters
top_day_clusters = df_daytime['hotspot_cluster'].value_counts().head(5)
print("Top Daytime Bottlenecks:\n", top_day_clusters)

Top Daytime Bottlenecks:
 hotspot_cluster
2     3426
3     1628
5      846
29     467
31     362
Name: count, dtype: int64


In [ ]:
# Ensure hour column exists from the timestamp
df['hour'] = pd.to_datetime(df['created_datetime']).dt.hour

# Define clean, non-overlapping 6-hour shift boundaries
df_morning   = df[(df['hour'] >= 8)  & (df['hour'] < 14) & (df['hotspot_cluster'] != -1)]
df_afternoon = df[(df['hour'] >= 14) & (df['hour'] < 20) & (df['hotspot_cluster'] != -1)]
df_evening   = df[(df['hour'] >= 20) | (df['hour'] < 2)  & (df['hotspot_cluster'] != -1)] # Uses OR (|) for midnight crossing
df_nighttime = df[(df['hour'] >= 2)  & (df['hour'] < 8)  & (df['hotspot_cluster'] != -1)]

# Print the top 10 hotspots for each distinct shift
print("=== TOP 10 MORNING HOTSPOTS (08:00 - 14:00) ===")
print(df_morning['hotspot_cluster'].value_counts().head(10))

print("\n=== TOP 10 AFTERNOON HOTSPOTS (14:00 - 20:00) ===")
print(df_afternoon['hotspot_cluster'].value_counts().head(10))

print("\n=== TOP 10 EVENING HOTSPOTS (20:00 - 02:00) ===")
print(df_evening['hotspot_cluster'].value_counts().head(10))

print("\n=== TOP 10 NIGHTTIME HOTSPOTS (02:00 - 08:00) ===")
print(df_nighttime['hotspot_cluster'].value_counts().head(10))

=== TOP 10 MORNING HOTSPOTS (08:00 - 14:00) ===
hotspot_cluster
3      711
2      635
7      226
5      204
13     166
17     158
4      157
167    111
52      93
44      92
Name: count, dtype: int64

=== TOP 10 AFTERNOON HOTSPOTS (14:00 - 20:00) ===
hotspot_cluster
2      2216
6       393
17      158
29      155
65      117
161     113
3        86
14       83
5        61
117      58
Name: count, dtype: int64

=== TOP 10 EVENING HOTSPOTS (20:00 - 02:00) ===
hotspot_cluster
2     10019
8      3698
29     3104
5      2955
3      2040
31     1432
15     1162
22     1004
17      880
21      866
Name: count, dtype: int64

=== TOP 10 NIGHTTIME HOTSPOTS (02:00 - 08:00) ===
hotspot_cluster
2      13601
3       7580
5       4296
29      2986
31      2512
8       2392
22      1143
112      843
25       757
13       712
Name: count, dtype: int64


In [ ]:
# Select the most critical unique clusters across all shifts
shift_leaders = [2, 3, 5, 6, 7, 8, 29, 31]

# Extract the definitive profiles for your presentation
leader_profiles = df[df['hotspot_cluster'].isin(shift_leaders)].groupby('hotspot_cluster').agg(
    total_violations=('hotspot_cluster', 'count'),
    primary_junction=('junction_name', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
    top_vehicle=('vehicle_type', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
    top_violation=('violation_type', lambda x: x.mode()[0] if not x.mode().empty else "Unknown"),
    top_location=('location', lambda x: x.mode()[0] if not x.mode().empty else "Unknown")
).sort_values(by='total_violations', ascending=False)

print(leader_profiles)

                 total_violations                  primary_junction  \
hotspot_cluster                                                       
2                           26471           BTP040 - Elite Junction   
3                           10417    BTP051 - Safina Plaza Junction   
5                            7516                       No Junction   
29                           6318  BTP020 - Hosahalli Metro Station   
8                            6217                       No Junction   
31                           4020                       No Junction   
6                            1189                       No Junction   
7                             855                       No Junction   

                top_vehicle      top_violation  \
hotspot_cluster                                  
2                   SCOOTER  ["WRONG PARKING"]   
3                   SCOOTER  ["WRONG PARKING"]   
5                       CAR  ["WRONG PARKING"]   
29                  SCOOTER     ["NO PA